# 03.1 — DESI DR1 BAO ΛCDM Fitting: Corrected Pipeline

**Author:** Cherian Parangot Ittyipe (FRAS 41871)  
**Thesis DOI:** https://doi.org/10.5281/zenodo.17681763  
**Reference:** DESI 2024 VI, arXiv:2404.03002 (Data Release 1)

---

Corrections applied over `03_desi_lcdm_fitting.ipynb`:

1. **DR1 labelling** — all references say DESI DR1, not DR2
2. **File stems** — exact filenames verified from disk
3. **Loader** — reads only column 1 (the value); ignores column 2 (quantity string)
4. **MAP** — minimises total negative log-posterior, not χ²_BAO alone
5. **rᵈ prior flag** — `USE_RD_PRIOR` controls Planck rᵈ prior; default False
6. **Ωm derivation** — derived from posterior samples, compared to paper value
7. **Convergence** — autocorrelation time is primary diagnostic

**Do not modify `03_desi_lcdm_fitting.ipynb` while its chain is running.**

## Cell 1 — Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import camb
import emcee
import corner
import scipy.optimize as opt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR  = Path('/Users/cherianpi/Desktop/WMAP+DESI')
BAO_DIR   = BASE_DIR / 'bao_data'
IMG_DIR   = BASE_DIR / 'Python files' / 'Images'
CHAIN_DIR = BASE_DIR / 'Python files' / 'chains'
IMG_DIR.mkdir(parents=True, exist_ok=True)
CHAIN_DIR.mkdir(parents=True, exist_ok=True)

# ── Key scientific flag ────────────────────────────────────────────────────
# False → DESI BAO-only (no external sound-horizon calibration)
# True  → DESI BAO + Planck rᵈ prior (label all outputs explicitly)
# For Notebook 04 (joint WMAP9+DESI) this MUST be False.
USE_RD_PRIOR = True

RD_MEAN = 147.09   # Mpc  (used only when USE_RD_PRIOR = True)
RD_SIG  = 0.26     # Mpc

# ── MCMC settings ──────────────────────────────────────────────────────────
NWALKERS = 32
NSTEPS   = 3000
NBURN    = 500

PARAM_NAMES  = [r'$\Omega_b h^2$', r'$\Omega_c h^2$', r'$H_0$']
PRIOR_BOUNDS = np.array([
    [0.005, 0.04],
    [0.05,  0.30],
    [50.0,  90.0],
])

label_suffix = '+ Planck rᵈ prior' if USE_RD_PRIOR else 'BAO-only'
print(f'Run mode  : DESI DR1 {label_suffix}')
print(f'Data dir  : {BAO_DIR}')

## Cell 2 — Load DESI DR1 BAO Data

File stems verified directly from disk on 2026-06-16.  
Each mean file has three columns: `z  value  quantity_name`.  
The loader reads only column index 1 (the numeric value).

In [ ]:
def load_mean(filepath):
    """
    Load BAO mean vector from a 3-column file: z  value  quantity_name.
    Returns a 1D array of the values (column 1 only).
    Skips comment lines (#) and any line that cannot be parsed.
    """
    values = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            try:
                values.append(float(parts[1]))   # column 1 = measurement value
            except (IndexError, ValueError):
                continue
    return np.array(values)


def load_cov(filepath):
    """Load covariance matrix. All entries are numeric."""
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            try:
                rows.append([float(x) for x in line.split()])
            except ValueError:
                continue
    return np.array(rows)


# ── DR1 tracer definitions — stems verified from disk ──────────────────────
# Source: arXiv:2404.03002 Table 1
# BGS  : isotropic  DV/rd
# LRG1,2, LRG3+ELG1, ELG2, Lya : anisotropic DM/rd + DH/rd
# QSO  : isotropic  DV/rd
TRACERS = {
    'BGS':       {
        'stem': 'desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4',
        'mode': 'iso',
        'zeff': 0.295
    },
    'LRG1':      {
        'stem': 'desi_2024_gaussian_bao_LRG_GCcomb_z0.4-0.6',
        'mode': 'aniso',
        'zeff': 0.510
    },
    'LRG2':      {
        'stem': 'desi_2024_gaussian_bao_LRG_GCcomb_z0.6-0.8',
        'mode': 'aniso',
        'zeff': 0.706
    },
    'LRG3+ELG1': {
        'stem': 'desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1',
        'mode': 'aniso',
        'zeff': 0.930
    },
    'ELG2':      {
        'stem': 'desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6',
        'mode': 'aniso',
        'zeff': 1.317
    },
    'QSO':       {
        'stem': 'desi_2024_gaussian_bao_QSO_GCcomb_z0.8-2.1',
        'mode': 'iso',
        'zeff': 1.491
    },
    'Lya':       {
        'stem': 'desi_2024_gaussian_bao_Lya_GCcomb',
        'mode': 'aniso',
        'zeff': 2.330
    },
}

bao_data = {}
n_total  = 0
for name, info in TRACERS.items():
    mean_file = BAO_DIR / f"{info['stem']}_mean.txt"
    cov_file  = BAO_DIR / f"{info['stem']}_cov.txt"
    if not mean_file.exists() or not cov_file.exists():
        missing = [s for s, f in [('mean', mean_file), ('cov', cov_file)] if not f.exists()]
        print(f'WARNING: {name:12s} — missing files: {missing}')
        continue
    mean_vec = load_mean(mean_file)
    cov_mat  = load_cov(cov_file)
    if mean_vec.size == 0 or cov_mat.size == 0:
        print(f'WARNING: {name:12s} — files found but loaded empty')
        continue
    bao_data[name] = {
        'mean': mean_vec,
        'icov': np.linalg.inv(cov_mat),
        'mode': info['mode'],
        'zeff': info['zeff'],
    }
    n_total += mean_vec.size
    print(f'Loaded {name:12s} | zeff={info["zeff"]:.3f} | '
          f'mode={info["mode"]:5s} | ndata={mean_vec.size} | values={np.round(mean_vec, 4)}')

print(f'\nLoaded {len(bao_data)} of {len(TRACERS)} tracers  |  Total measurements: {n_total}')

In [ ]:
## Cell 2 — Load DESI DR1 BAO Data
#
# File stems verified directly from disk on 2026-06-16.
# Each mean file has three columns: z  value  quantity_name.
#
# FIX (2026-06-27): the original loader read only column 1 (the value)
# and discarded column 2 (the quantity label), assuming every aniso
# tracer stores [DM_over_rs, DH_over_rs] in that row order. The Lya
# file stores them in the OPPOSITE order, which silently swapped DM
# and DH for Lya and inflated the BAO chi^2 by a factor of ~600
# (confirmed against the original bug in 04_wmap9_desi_joint_lcdm.ipynb).
# This version reads the label and assigns by name, so it's correct
# regardless of row order in any tracer file.

def load_mean_labeled(filepath):
    """Returns {quantity_name: value}, immune to row ordering in the file."""
    out = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            try:
                out[parts[2]] = float(parts[1])
            except ValueError:
                continue
    return out


def load_cov(filepath):
    """Load covariance matrix. All entries are numeric."""
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            try:
                rows.append([float(x) for x in line.split()])
            except ValueError:
                continue
    return np.array(rows)


# ── DR1 tracer definitions — stems verified from disk ──────────────────────
# Source: arXiv:2404.03002 Table 1
# BGS  : isotropic  DV/rd
# LRG1,2, LRG3+ELG1, ELG2, Lya : anisotropic DM/rd + DH/rd
# QSO  : isotropic  DV/rd
TRACERS = {
    'BGS':       {
        'stem': 'desi_2024_gaussian_bao_BGS_BRIGHT-21.5_GCcomb_z0.1-0.4',
        'mode': 'iso',
        'zeff': 0.295
    },
    'LRG1':      {
        'stem': 'desi_2024_gaussian_bao_LRG_GCcomb_z0.4-0.6',
        'mode': 'aniso',
        'zeff': 0.510
    },
    'LRG2':      {
        'stem': 'desi_2024_gaussian_bao_LRG_GCcomb_z0.6-0.8',
        'mode': 'aniso',
        'zeff': 0.706
    },
    'LRG3+ELG1': {
        'stem': 'desi_2024_gaussian_bao_LRG+ELG_LOPnotqso_GCcomb_z0.8-1.1',
        'mode': 'aniso',
        'zeff': 0.930
    },
    'ELG2':      {
        'stem': 'desi_2024_gaussian_bao_ELG_LOPnotqso_GCcomb_z1.1-1.6',
        'mode': 'aniso',
        'zeff': 1.317
    },
    'QSO':       {
        'stem': 'desi_2024_gaussian_bao_QSO_GCcomb_z0.8-2.1',
        'mode': 'iso',
        'zeff': 1.491
    },
    'Lya':       {
        'stem': 'desi_2024_gaussian_bao_Lya_GCcomb',
        'mode': 'aniso',
        'zeff': 2.330
    },
}

REQUIRED_KEYS = {
    'aniso': ('DM_over_rs', 'DH_over_rs'),
    'iso':   ('DV_over_rs',),
}

bao_data = {}
n_total  = 0
for name, info in TRACERS.items():
    mean_file = BAO_DIR / f"{info['stem']}_mean.txt"
    cov_file  = BAO_DIR / f"{info['stem']}_cov.txt"
    if not mean_file.exists() or not cov_file.exists():
        missing = [s for s, f in [('mean', mean_file), ('cov', cov_file)] if not f.exists()]
        print(f'WARNING: {name:12s} — missing files: {missing}')
        continue

    labeled = load_mean_labeled(mean_file)
    cov_mat = load_cov(cov_file)

    needed  = REQUIRED_KEYS[info['mode']]
    missing_keys = [k for k in needed if k not in labeled]
    if missing_keys:
        print(f'WARNING: {name:12s} — missing expected quantities {missing_keys} '
              f'— found {list(labeled.keys())}')
        continue

    mean_vec = np.array([labeled[k] for k in needed])

    if mean_vec.size == 0 or cov_mat.size == 0:
        print(f'WARNING: {name:12s} — files found but loaded empty')
        continue

    bao_data[name] = {
        'mean': mean_vec,
        'icov': np.linalg.inv(cov_mat),
        'mode': info['mode'],
        'zeff': info['zeff'],
    }
    n_total += mean_vec.size
    print(f'Loaded {name:12s} | zeff={info["zeff"]:.3f} | '
          f'mode={info["mode"]:5s} | ndata={mean_vec.size} | values={np.round(mean_vec, 4)}')

print(f'\nLoaded {len(bao_data)} of {len(TRACERS)} tracers  |  Total measurements: {n_total}')

## Cell 3 — CAMB Theory Engine

In [ ]:
def get_bao_theory(ombh2, omch2, H0, zeff, mode):
    """
    Compute BAO observables at zeff.
    Returns ([DM/rd, DH/rd], rd) for mode='aniso'
    Returns ([DV/rd],        rd) for mode='iso'
    """
    pars = camb.CAMBparams()
    pars.set_cosmology(
        H0=H0, ombh2=ombh2, omch2=omch2,
        mnu=0.06, omk=0.0, tau=0.054
    )
    pars.set_dark_energy()
    pars.InitPower.set_params(As=2.1e-9, ns=0.965)
    pars.set_for_lmax(500, lens_potential_accuracy=0)
    results = camb.get_results(pars)

    rd  = results.get_derived_params()['rdrag']
    DA  = results.angular_diameter_distance(zeff)
    DM  = DA * (1.0 + zeff)
    H_z = results.hubble_parameter(zeff)
    DH  = 299792.458 / H_z
    DV  = (zeff * DM**2 * DH)**(1.0 / 3.0)

    if mode == 'aniso':
        return np.array([DM / rd, DH / rd]), rd
    else:
        return np.array([DV / rd]), rd


def get_rd_camb(ombh2, omch2, H0):
    pars = camb.CAMBparams()
    pars.set_cosmology(
        H0=H0, ombh2=ombh2, omch2=omch2,
        mnu=0.06, omk=0.0, tau=0.054
    )
    pars.set_for_lmax(500, lens_potential_accuracy=0)
    return camb.get_results(pars).get_derived_params()['rdrag']


# Sanity check against DR1 ALL file values
th, rd = get_bao_theory(0.0224, 0.120, 67.4, 0.510, 'aniso')
print(f'Sanity check — LRG1 (z=0.510, aniso):')
print(f'  rᵈ    = {rd:.3f} Mpc       (expect ~147)')
print(f'  DM/rᵈ = {th[0]:.4f}         (DR1 file: 13.6200)')
print(f'  DH/rᵈ = {th[1]:.4f}         (DR1 file: 20.9833)')
th2, _ = get_bao_theory(0.0224, 0.120, 67.4, 1.491, 'iso')
print(f'Sanity check — QSO  (z=1.491, iso):')
print(f'  DV/rᵈ = {th2[0]:.4f}         (DR1 file: 26.0722)')

## Cell 4 — Likelihood, Prior, Posterior

In [ ]:
def log_like_bao(theta):
    """Gaussian BAO likelihood over all loaded DESI DR1 tracers."""
    ombh2, omch2, H0 = theta
    lnL = 0.0
    for name, d in bao_data.items():
        try:
            theory, _ = get_bao_theory(ombh2, omch2, H0, d['zeff'], d['mode'])
        except Exception:
            return -np.inf
        delta = d['mean'] - theory
        lnL  -= 0.5 * delta @ d['icov'] @ delta
    return lnL


def log_prior(theta):
    ombh2, omch2, H0 = theta
    if (ombh2 < PRIOR_BOUNDS[0, 0] or ombh2 > PRIOR_BOUNDS[0, 1] or
        omch2 < PRIOR_BOUNDS[1, 0] or omch2 > PRIOR_BOUNDS[1, 1] or
        H0    < PRIOR_BOUNDS[2, 0] or H0    > PRIOR_BOUNDS[2, 1]):
        return -np.inf
    if not USE_RD_PRIOR:
        return 0.0
    try:
        rd = get_rd_camb(ombh2, omch2, H0)
    except Exception:
        return -np.inf
    return -0.5 * ((rd - RD_MEAN) / RD_SIG)**2


def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = log_like_bao(theta)
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


print(f'Posterior defined. Run mode: DESI DR1 {label_suffix}')

## Cell 5 — MAP Fit (Corrected)

Minimises total negative log-posterior, not χ²_BAO alone.

In [ ]:
def neg_log_posterior(theta):
    val = log_posterior(theta)
    return 1e10 if not np.isfinite(val) else -val


theta0 = np.array([0.0224, 0.120, 67.4])
print('Running MAP optimisation...')
result = opt.minimize(
    neg_log_posterior, theta0, method='Nelder-Mead',
    options=dict(xatol=1e-5, fatol=1e-4, maxiter=5000, disp=True)
)
theta_map   = result.x
h_map       = theta_map[2] / 100.0
Omega_m_map = (theta_map[0] + theta_map[1]) / h_map**2
rd_map      = get_rd_camb(*theta_map)

print(f'\nMAP — DESI DR1 {label_suffix}:')
print(f'  Ωb h²  = {theta_map[0]:.5f}')
print(f'  Ωc h²  = {theta_map[1]:.5f}')
print(f'  H₀     = {theta_map[2]:.3f}  km/s/Mpc')
print(f'  Ωm     = {Omega_m_map:.4f}  (paper: 0.295 ± 0.015)')
print(f'  rᵈ     = {rd_map:.3f}  Mpc')
print(f'  rᵈ h   = {rd_map * h_map:.3f}  Mpc  (paper: 101.8 ± 1.3)')

## Cell 6 — MCMC Sampling

In [ ]:
ndim = 3
rng  = np.random.default_rng(42)
p0   = theta_map + 1e-3 * rng.standard_normal((NWALKERS, ndim))
for i in range(ndim):
    p0[:, i] = np.clip(p0[:, i],
                       PRIOR_BOUNDS[i, 0] + 1e-4,
                       PRIOR_BOUNDS[i, 1] - 1e-4)

sampler = emcee.EnsembleSampler(
    NWALKERS, ndim, log_posterior,
    moves=emcee.moves.DEMove()
)
print(f'Running MCMC: {NWALKERS} walkers × {NSTEPS} steps — DESI DR1 {label_suffix}')
sampler.run_mcmc(p0, NSTEPS, progress=True)
print('Done.')

chain_file = CHAIN_DIR / f'desi_dr1_lcdm_{"with_rd_prior" if USE_RD_PRIOR else "bao_only"}_chains.npy'
np.save(chain_file, sampler.get_chain())
print(f'Chain saved: {chain_file}')

## Cell 7 — Convergence Diagnostics

In [ ]:
try:
    tau = sampler.get_autocorr_time(quiet=True)
    print('Autocorrelation time (primary diagnostic):')
    for pname, t in zip(['ombh2', 'omch2', 'H0'], tau):
        ratio  = NSTEPS / t
        status = 'CONVERGED' if ratio > 50 else ('OK (preliminary)' if ratio > 20 else 'NOT CONVERGED')
        print(f'  τ({pname:6s}) = {t:6.1f}   NSTEPS/τ = {ratio:5.1f}   [{status}]')
    NBURN_AUTO = max(NBURN, int(2 * np.max(tau)))
    print(f'\nBurn-in: {NBURN_AUTO} steps')
except Exception as e:
    print(f'Autocorrelation failed: {e}')
    NBURN_AUTO = NBURN

chain     = sampler.get_chain()
post_burn = chain[NBURN_AUTO:]
n_groups  = 4
group_sz  = NWALKERS // n_groups
print('\nApproximate G-R R̂ (walker groups — use τ above as primary):')
for i, pname in enumerate(['ombh2', 'omch2', 'H0']):
    gm   = [post_burn[:, j*group_sz:(j+1)*group_sz, i].mean() for j in range(n_groups)]
    gv   = [post_burn[:, j*group_sz:(j+1)*group_sz, i].var()  for j in range(n_groups)]
    W    = np.mean(gv)
    B    = np.var(gm) * len(post_burn)
    n    = len(post_burn)
    Rhat = np.sqrt(((n-1)/n * W + B/n) / W) if W > 0 else np.nan
    print(f'  R̂({pname:6s}) = {Rhat:.4f}  [{"OK" if Rhat < 1.05 else "check"}]')

## Cell 8 — Posterior Summary and Ωm

In [ ]:
flat_samples = sampler.get_chain(discard=NBURN_AUTO, flat=True)
print(f'Flat samples: {flat_samples.shape[0]} (after {NBURN_AUTO} burn-in steps)\n')

print(f'=== DESI DR1 ΛCDM Posterior ({label_suffix}) ===')
print(f'{"Parameter":<12}  {"Median":>10}  {"Std":>10}  {"16th":>10}  {"84th":>10}')
print('-' * 58)
for i, pname in enumerate(['ombh2', 'omch2', 'H0']):
    s = flat_samples[:, i]
    q16, q50, q84 = np.percentile(s, [16, 50, 84])
    print(f'{pname:<12}  {q50:10.5f}  {s.std():10.5f}  {q16:10.5f}  {q84:10.5f}')

print('\n=== Derived Parameters ===')
ombh2_s   = flat_samples[:, 0]
omch2_s   = flat_samples[:, 1]
H0_s      = flat_samples[:, 2]
h_s       = H0_s / 100.0
Omega_m_s = (ombh2_s + omch2_s) / h_s**2
q16, q50, q84 = np.percentile(Omega_m_s, [16, 50, 84])
print(f'Ωm = {q50:.4f} +{q84-q50:.4f} -{q50-q16:.4f}')
print(f'     DESI DR1 paper (BAO-only): 0.295 ± 0.015')

rd_s  = np.array([get_rd_camb(ob, oc, H)
                  for ob, oc, H in zip(ombh2_s[::10], omch2_s[::10], H0_s[::10])])
rdh_s = rd_s * (H0_s[::10] / 100.0)
q16r, q50r, q84r = np.percentile(rdh_s, [16, 50, 84])
print(f'rᵈh  = {q50r:.2f} +{q84r-q50r:.2f} -{q50r-q16r:.2f}  Mpc')
print(f'       DESI DR1 paper (BAO-only): 101.8 ± 1.3 Mpc')

## Cell 9 — Corner Plot

In [ ]:
fig = corner.corner(
    flat_samples,
    labels=PARAM_NAMES,
    quantiles=[0.16, 0.50, 0.84],
    show_titles=True,
    title_kwargs={'fontsize': 10},
    smooth=1.0, smooth1d=1.0,
    truths=theta_map
)
fig.suptitle(f'DESI DR1 ΛCDM Posterior — {label_suffix}', y=1.01, fontsize=12)
fname = f'desi_dr1_corner_{"with_rd_prior" if USE_RD_PRIOR else "bao_only"}.png'
fig.savefig(IMG_DIR / fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {IMG_DIR / fname}')

## Cell 10 — Distance Ladder Plot vs DR1 Table 1 Data Points

In [ ]:
# Hardcoded from arXiv:2404.03002 Table 1
desi_table = {
    'BGS':       {'z': 0.295, 'DV_rd': 7.93,  'DV_rd_err': 0.15,  'mode': 'iso'},
    'LRG1':      {'z': 0.510, 'DM_rd': 13.62, 'DM_rd_err': 0.25,
                              'DH_rd': 20.98,  'DH_rd_err': 0.61,  'mode': 'aniso'},
    'LRG2':      {'z': 0.706, 'DM_rd': 16.85, 'DM_rd_err': 0.32,
                              'DH_rd': 20.08,  'DH_rd_err': 0.60,  'mode': 'aniso'},
    'LRG3+ELG1': {'z': 0.930, 'DM_rd': 21.71, 'DM_rd_err': 0.28,
                              'DH_rd': 17.88,  'DH_rd_err': 0.35,  'mode': 'aniso'},
    'ELG2':      {'z': 1.317, 'DM_rd': 27.79, 'DM_rd_err': 0.69,
                              'DH_rd': 13.82,  'DH_rd_err': 0.42,  'mode': 'aniso'},
    'QSO':       {'z': 1.491, 'DV_rd': 26.07, 'DV_rd_err': 0.67,  'mode': 'iso'},
    'Lya':       {'z': 2.330, 'DM_rd': 39.71, 'DM_rd_err': 0.94,
                              'DH_rd': 8.52,   'DH_rd_err': 0.17,  'mode': 'aniso'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
z_fine = np.linspace(0.1, 2.6, 200)
DV_theory, DMDH_theory, z_plot = [], [], []
for z in z_fine:
    try:
        ta, rd = get_bao_theory(*theta_map, z, 'aniso')
        ti, _  = get_bao_theory(*theta_map, z, 'iso')
        DV_theory.append(ti[0] / z**(2/3))
        DMDH_theory.append(ta[0] / ta[1] / z)
        z_plot.append(z)
    except Exception:
        continue

axes[0].plot(z_plot, DV_theory,   'k-', lw=1.5, label='MAP ΛCDM')
axes[1].plot(z_plot, DMDH_theory, 'k-', lw=1.5, label='MAP ΛCDM')

for name, d in desi_table.items():
    z = d['z']
    if d['mode'] == 'iso':
        axes[0].errorbar(z, d['DV_rd']/z**(2/3), yerr=d['DV_rd_err']/z**(2/3),
                         fmt='o', capsize=4, label=name)
    else:
        axes[1].errorbar(z, d['DM_rd']/(d['DH_rd']*z),
                         yerr=np.sqrt((d['DM_rd_err']/d['DH_rd']/z)**2 +
                                      (d['DM_rd']*d['DH_rd_err']/d['DH_rd']**2/z)**2),
                         fmt='o', capsize=4, label=name)

axes[0].set(xlabel='Redshift z', ylabel=r'$D_V/(r_d z^{2/3})$',
            title=f'DESI DR1 BAO — {label_suffix}')
axes[0].legend(fontsize=8)
axes[1].set(xlabel='Redshift z', ylabel=r'$D_M/(D_H\cdot z)$',
            title=f'DESI DR1 BAO — {label_suffix}')
axes[1].legend(fontsize=8)
plt.tight_layout()
fname = f'desi_dr1_distance_ladder_{"with_rd_prior" if USE_RD_PRIOR else "bao_only"}.png'
plt.savefig(IMG_DIR / fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {IMG_DIR / fname}')

## Cell 11 — Notes for Notebook 04

```
JOINT WMAP9 + DESI RULES
─────────────────────────
1. USE_RD_PRIOR = False — WMAP9 provides early-universe calibration.
   Adding Planck rᵈ prior would double-count early-universe information.

2. lnL_total = lnL_WMAP9(θ) + lnL_DESI_BAO(θ)
   θ = [ombh2, omch2, H0, ns, ln(10^10 As), tau]
   BAO constrains only ombh2, omch2, H0.
   ns, As, tau constrained by WMAP9 only.

3. Comparison targets:
   DESI DR1 BAO-only : Ωm = 0.295 ± 0.015, rdh = 101.8 ± 1.3 Mpc
   DESI DR1 + CMB    : Ωm = 0.307 ± 0.005, H0  = 67.97 ± 0.38
   Planck 2018       : H0 = 67.36 ± 0.54
   WMAP9 alone       : H0 = 69.32 ± 0.80

4. log_like_bao(theta) from Cell 4 can be imported directly into 04.
```